In [264]:
import csv
import re
import bisect

In [265]:
def incarca_atomi(fisier_csv):
    atomi = {}
    with open(fisier_csv, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            cod_str = row['cod'].strip()
            atom = row['atom'].strip()
            if cod_str: 
                cod = int(cod_str)
                atomi[atom] = cod
            else:
                print(f"Cod gol pentru atomul '{atom}'")
    return atomi

### Generare TS fara reguli

In [266]:
class TabelSimboluri:
    def __init__(self):
        self.simboluri = []

    def pozitie(self, simbol):
        if simbol not in self.simboluri:
            self.simboluri.append(simbol)
        return self.simboluri.index(simbol)

    def scrie_in_fisier(self, nume_fisier):
        with open(nume_fisier, 'w') as f:
            for i, s in enumerate(self.simboluri):
                f.write(f"{i},{s}\n")

### Generare TS cu ordine lexicografica

In [267]:
class TabelSimboluriOrdonat:
    def __init__(self):
        self.simboluri = []
        self.dictionar_simboluri = {}

    def pozitie(self, simbol):
        if simbol in self.dictionar_simboluri:
            return self.dictionar_simboluri[simbol]
        
        bisect.insort(self.simboluri, simbol)
        
        self.dictionar_simboluri = {s: i for i, s in enumerate(self.simboluri)}

        return self.dictionar_simboluri[simbol]

    def scrie_in_fisier(self, nume_fisier):
        with open(nume_fisier, 'w') as f:
            for i, s in enumerate(self.simboluri):
                f.write(f"{i},{s}\n")

### Generare FIP (versiune 4 curs)

In [268]:
def analizor_lexical(text, atomi, ts):
    fip = []

    cod_ID = atomi.get("ID",100)
    cod_CONST = atomi.get("CONST",101)
    cod_virgula = 8

    pattern = r"(<-|==|!=|<=|>=|<<|>>|[{}();,+\-*:%<>#]|[A-Za-z_][A-Za-z0-9_]*|[0-9]+)"
    tokens = re.findall(pattern, text)

    for atom in tokens:
        # daca atom este cuv-cheie / operator / separator
        if atom in atomi:
            fip.append((atom, atomi[atom], 0))
        elif atom == ',':
            fip.append((atom, cod_virgula, 0))
        else:
            # daca atom este identificator
            if re.match(r'^[A-Za-z_][A-Za-z0-9_]*$', atom):
                pos = ts.pozitie(atom)
                fip.append((atom, cod_ID, pos))

            # daca atom este constanta numerica
            elif re.match(r'^[0-9]+$', atom):
                poz = ts.pozitie(atom)
                fip.append((atom, cod_CONST, poz))
            
            else:
                print(f"Atom necunoscut: {atom}")

    return fip

def afisare_fip(fip, nume_fisier):
    with open(nume_fisier,'r+') as file:
        file.truncate(0)

    with open(nume_fisier, 'w') as f:
        
        for atom, cod, poz in fip:
            f.write(f"{atom}, {cod}, {poz}\n")

### Generare FIP cu TS separate

In [269]:
def analizor_lexical_restrictii(text, atomi, ts_id, ts_const):
    fip = []

    cod_ID = atomi.get("ID",100)
    cod_CONST = atomi.get("CONST",101)
    cod_virgula = 8

    pattern = r"(<-|==|!=|<=|>=|>>|<<|[{}()]|[;,+\-*:%<>#]+|[A-Za-z_][A-Za-z0-9_]*|[0-9]+)"

    linii = text.splitlines()
    for nr_linie, linie in enumerate(linii, start = 1):
        tokens = re.findall(pattern, linie)

        for i, atom in enumerate(tokens):

            if atom in atomi:
                fip.append((atom, atomi[atom], 0))

            elif atom == ',':
                fip.append((atom, cod_virgula, 0))

            elif re.match(r'^[0-9]+$', atom):
                poz = ts_const.pozitie(atom)
                fip.append((atom, cod_CONST, poz))

            elif re.match(r'^[A-Za-z_][A-Za-z0-9_]*$', atom):
                if i + 1 < len(tokens) and tokens[i + 1] in {">>", "<<"}:
                    print(f"Eroare lexicala pe linia {nr_linie}. Atom necunoscut: {atom}")
                elif  i + 2 < len(tokens) and tokens[i + 2] in {">>", "<<"} and tokens[i + 1] == " ":
                    print(f"Eroare lexicala pe linia {nr_linie}. Atom necunoscut: {atom}")
                else:
                    pos = ts_id.pozitie(atom)
                    fip.append((atom, cod_ID, pos))

            else:
                print(f"Eroare lexicala pe linia {nr_linie}. Atom necunoscut: {atom}")
        
        rest = re.sub(pattern, '', linie).strip()
        if rest:
            print(f"Eroare lexicala pe linia {nr_linie}. Secventa necunoscuta: {rest}")

    return fip

In [270]:

def main():
    atomi = incarca_atomi("atom.csv")
    ts = TabelSimboluri()

    with open("input.txt", "r", encoding="utf-8") as f:
        text = f.read()

    fip = analizor_lexical(text, atomi, ts)
    afisare_fip(fip, "FIP.txt")
    ts.scrie_in_fisier("TS.txt")

In [271]:
def main_restrictii():
    atomi = incarca_atomi("atom.csv")
    ts_id = TabelSimboluriOrdonat()
    ts_const = TabelSimboluriOrdonat()
    
    with open("input_eroare.txt", "r", encoding="utf-8") as f:
        text = f.read()

    fip = analizor_lexical_restrictii(text, atomi, ts_id, ts_const)
    afisare_fip(fip, "FIP.txt")
    ts_id.scrie_in_fisier("TS_ID.txt")
    ts_const.scrie_in_fisier("TS_CONST.txt")


In [272]:
main_restrictii()

Eroare lexicala pe linia 5. Secventa necunoscuta: =
Eroare lexicala pe linia 6. Atom necunoscut: cin
Eroare lexicala pe linia 7. Atom necunoscut: ++
